In [3]:
import pandas as pd
data = pd.read_csv('./data/train.csv')

In [4]:
data

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [66]:
import torch

features = ['HomePlanet', 'CryoSleep', 'RoomNumber', 'Deck', 'Side', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
output = ['Transported']

data = pd.read_csv('./data/train.csv')


# Formatting data
home_planet_category = pd.Categorical(data['HomePlanet'], ordered=False)
data['HomePlanet'] = home_planet_category.codes

destination_category = pd.Categorical(data['Destination'], ordered=False)
data['Destination'] = destination_category.codes

def format (x):
  if pd.isnull(x):
    return [pd.NA, pd.NA, pd.NA]
  else:
    return x.split('/')
data[['Deck', 'RoomNumber', 'Side']] = pd.DataFrame([format(x) for x in data['Cabin'].tolist() ])

deck_category = pd.Categorical(data['Deck'], ordered=False)
data['Deck'] = deck_category.codes
side_category = pd.Categorical(data['Side'], ordered=False)
data['Side'] = side_category.codes
data = data.drop('Cabin', axis=1)

# dropping nan s
data = data.dropna()
data['CryoSleep'] = data['CryoSleep'].astype('bool')
data['VIP'] = data['VIP'].astype('bool')
data['RoomNumber'] = data['RoomNumber'].astype('int8')

X = torch.tensor(data[features].to_numpy().tolist())

Y = torch.tensor(data['Transported'].tolist())

print(X[1:10], Y[1:10])


tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 5.0000e+00, 1.0000e+00, 2.0000e+00,
         2.4000e+01, 0.0000e+00, 1.0900e+02, 9.0000e+00, 2.5000e+01, 5.4900e+02,
         4.4000e+01],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 2.0000e+00,
         5.8000e+01, 1.0000e+00, 4.3000e+01, 3.5760e+03, 0.0000e+00, 6.7150e+03,
         4.9000e+01],
        [1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 2.0000e+00,
         3.3000e+01, 0.0000e+00, 0.0000e+00, 1.2830e+03, 3.7100e+02, 3.3290e+03,
         1.9300e+02],
        [0.0000e+00, 0.0000e+00, 1.0000e+00, 5.0000e+00, 1.0000e+00, 2.0000e+00,
         1.6000e+01, 0.0000e+00, 3.0300e+02, 7.0000e+01, 1.5100e+02, 5.6500e+02,
         2.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 5.0000e+00, 0.0000e+00, 1.0000e+00,
         4.4000e+01, 0.0000e+00, 0.0000e+00, 4.8300e+02, 0.0000e+00, 2.9100e+02,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 2.0000e+00, 5.0000e+00, 1.0000e+00, 2.0000e+00,

In [73]:
X.size()[1]

13

In [77]:
from torch import nn

class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_1 = nn.Linear(X.size()[1], 1)
    
  def forward(self, x):
    return self.layer_1(x)

model = Model()

In [81]:

loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
for t in range(2000):
  y_pred = model(X[t])
  y_loss = loss(y_pred, Y[t])
  print(y_loss)
  print(Y[t])
  optimizer.zero_grad()
  y_loss.backward()
  optimizer.step()

print(f'Result: {model.string()}')

tensor(10.1987, grad_fn=<MseLossBackward0>)
tensor(False)


RuntimeError: Found dtype Bool but expected Float